In [1]:
%pip install pdf2image pytesseract PyPDF2 pdfminer.six Pillow regex pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 1 — Config for your current VS Code structure
from pathlib import Path

# You're inside: EarDisableResearch/Models/data/NLP.ipynb
# Go up two folders to reach the project root
ROOT = Path.cwd().parents[1]   # from Models/data → EarDisableResearch

# Correct path to your PDF
PDF_PATH = ROOT / "Models" / "data" / "raw" / "OUT SOURCING PART 1.pdf"

# Output directory for processed OCR and text
OUT_DIR = ROOT / "Models" / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ PDF path:", PDF_PATH)
print("✅ Output directory:", OUT_DIR)
print("✅ Exists:", PDF_PATH.exists())


✅ PDF path: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\Models\data\raw\OUT SOURCING PART 1.pdf
✅ Output directory: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\Models\data\processed
✅ Exists: False


In [3]:
# cell: find_pdf_and_set_paths
from pathlib import Path
import sys, pprint

FILENAME = "OUT SOURCING PART 1.pdf"

# Choose a safe project root: two parents up if notebook inside Models/data,
# otherwise current working directory. This tries both heuristics.
cwd = Path.cwd()
candidates = [cwd, cwd.parents[0], cwd.parents[1], cwd.parents[2]]
# make unique while preserving order
seen = set()
roots = []
for c in candidates:
    if c not in seen:
        roots.append(c)
        seen.add(c)

# also add the absolute repo root guess (topmost)
roots.append(Path.cwd().anchor)  # e.g., "C:\\" on Windows (only used if no match)

found = []
for root in roots:
    # do a limited-depth search (depth=4) to avoid scanning whole disk accidentally
    for p in root.rglob(FILENAME):
        found.append(p.resolve())
    if found:
        break

# If still nothing, try scanning entire current working directory tree (may take time)
if not found:
    print("No PDF found in quick search under heuristics. Doing full workspace scan (may take a while)...")
    for p in cwd.rglob(FILENAME):
        found.append(p.resolve())

if not found:
    print("No matches found for", FILENAME)
    print("Current working directory:", cwd)
    print("Please right-click the PDF in VS Code Explorer → 'Copy Path' and paste it below, or move the PDF into a known folder (e.g., Data/raw).")
    # print the top-level folders to help locate
    print("\nTop-level folders in workspace:")
    pprint.pprint(sorted([x.name for x in cwd.iterdir() if x.is_dir()])[:50])
    # set placeholders so subsequent cells won't crash
    PDF_PATH = None
    OUT_DIR = cwd / "Data" / "processed"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    print(f"Found {len(found)} match(es). Using the first one.")
    for i, p in enumerate(found[:10], start=1):
        print(f"  [{i}] {p}")
    PDF_PATH = found[0]
    # set OUT_DIR near the found file (Data/processed sibling) if possible; else use cwd/processed
    # attempt to find a 'Data' ancestor; otherwise use parent()/processed
    data_ancestor = None
    for a in PDF_PATH.parents:
        if a.name.lower() == "data" or a.name.lower() == "raw":
            data_ancestor = a.parent if a.name.lower()=="raw" else a
            break
    if data_ancestor:
        OUT_DIR = data_ancestor / "processed"
    else:
        OUT_DIR = PDF_PATH.parent.parent / "processed"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print("\nSet PDF_PATH to:", PDF_PATH)
    print("Set OUT_DIR to:", OUT_DIR)
    print("PDF exists:", PDF_PATH.exists())

# export these for notebook namespace
globals().update({"PDF_PATH": PDF_PATH, "OUT_DIR": OUT_DIR})


Found 1 match(es). Using the first one.
  [1] C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\raw\OUT SOURCING PART 1.pdf

Set PDF_PATH to: C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\raw\OUT SOURCING PART 1.pdf
Set OUT_DIR to: C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed
PDF exists: True


In [4]:
# Cell 2: text_extraction
from PyPDF2 import PdfReader
from pdfminer.high_level import extract_text
import json

reader = PdfReader(str(PDF_PATH))
n_pages = len(reader.pages)
page_texts = []

for i in range(n_pages):
    txt = ""
    try:
        txt = reader.pages[i].extract_text() or ""
    except Exception:
        txt = ""
    if not txt.strip():
        try:
            txt = extract_text(str(PDF_PATH), page_numbers=[i]) or ""
        except Exception as e:
            print(f"pdfminer failed on page {i+1}: {e}")
            txt = ""
    page_texts.append(txt)

with open(OUT_DIR / "pdf_page_texts.json", "w", encoding="utf-8") as f:
    json.dump({"n_pages": n_pages, "page_texts": page_texts}, f, ensure_ascii=False, indent=2)

print(f"Saved page_texts.json with {n_pages} pages.")


Saved page_texts.json with 12 pages.


In [5]:
# Cell B — Build page index
import json, re

with open(OUT_DIR / "pdf_page_texts.json", "r", encoding="utf-8") as f:
    data = json.load(f)
page_texts = data["page_texts"]
n_pages = data["n_pages"]

patterns = [r'kriyakarama', r'ක්\s*රියා', r'ක්‍රියා', r'ක්‍රියාකර', r'ක්‍රියාව']
pattern = re.compile("|".join(patterns), re.IGNORECASE)

kriyakarama_pages, text_extractable, need_ocr = [], [], []
for i, txt in enumerate(page_texts, start=1):
    if txt and txt.strip():
        text_extractable.append(i)
        if pattern.search(txt):
            kriyakarama_pages.append(i)
    else:
        need_ocr.append(i)

index = {
    "n_pages": n_pages,
    "kriyakarama_pages_found_by_text": kriyakarama_pages,
    "text_extractable_pages": text_extractable,
    "pages_needing_ocr": need_ocr
}
with open(OUT_DIR / "pdf_page_index.json", "w", encoding="utf-8") as f:
    json.dump(index, f, ensure_ascii=False, indent=2)

print("Index saved at:", OUT_DIR / "pdf_page_index.json")
print("kriyakarama_pages:", kriyakarama_pages)
print("pages_needing_ocr:", need_ocr)


Index saved at: C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed\pdf_page_index.json
kriyakarama_pages: []
pages_needing_ocr: [2, 4, 5]


In [6]:
%pip install pymupdf


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# CELL 0 — Install required Python packages (run once in your .venv)
# Run this from a notebook cell with a leading ! or in your terminal (recommended in terminal).
%pip install pymupdf easyocr pillow numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Cell C — OCR with Tesseract
import fitz              # pip install PyMuPDF
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract, io, json

# Windows: uncomment and set your tesseract.exe path if needed:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

with open(OUT_DIR / "pdf_page_index.json", "r", encoding="utf-8") as f:
    idx = json.load(f)

pages_to_ocr = set(idx.get("pages_needing_ocr", []))
# include neighbor pages around any text-detected kriyakarama pages
for p in idx.get("kriyakarama_pages_found_by_text", []):
    pages_to_ocr.update(range(max(1, p-1), min(idx["n_pages"]+1, p+2)))
pages_to_ocr = sorted(pages_to_ocr)
print("Pages selected for OCR:", pages_to_ocr)

doc = fitz.open(str(PDF_PATH))
ocr_results = {}
tess_lang = "sin+eng"   # requires sin.traineddata and eng; use "sin" if only Sinhala
tess_config = "--psm 6"

def preprocess(img):
    gray = img.convert("L")
    enh = ImageEnhance.Contrast(gray).enhance(1.4)
    return enh.filter(ImageFilter.MedianFilter(size=3))

if not pages_to_ocr:
    print("No pages marked for OCR. If kriyakarama headings are images, consider adding pages manually.")
else:
    for pnum in pages_to_ocr:
        page = doc.load_page(pnum - 1)
        pix = page.get_pixmap(dpi=300)
        pil_img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        pil_img_proc = preprocess(pil_img)
        try:
            text = pytesseract.image_to_string(pil_img_proc, lang=tess_lang, config=tess_config)
        except Exception as e:
            print(f"pytesseract error with lang={tess_lang}: {e} — falling back to eng")
            text = pytesseract.image_to_string(pil_img_proc, lang="eng", config=tess_config)
        text = text.replace("\r", "")
        ocr_results[str(pnum)] = text.strip()
        # save image + raw OCR txt
        pil_img.save(OUT_DIR / f"page_{pnum}.png")
        (OUT_DIR / f"ocr_page_{pnum}.txt").write_text(text, encoding="utf-8")
        print(f"OCRed page {pnum}: {len(text)} chars; saved page_{pnum}.png and ocr_page_{pnum}.txt")

    with open(OUT_DIR / "ocr_results_tesseract.json", "w", encoding="utf-8") as f:
        json.dump(ocr_results, f, ensure_ascii=False, indent=2)
    print("Saved OCR results to:", OUT_DIR / "ocr_results_tesseract.json")


Pages selected for OCR: [2, 4, 5]
OCRed page 2: 1016 chars; saved page_2.png and ocr_page_2.txt
OCRed page 4: 921 chars; saved page_4.png and ocr_page_4.txt
OCRed page 5: 497 chars; saved page_5.png and ocr_page_5.txt
Saved OCR results to: C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed\ocr_results_tesseract.json


In [9]:
# Cell D — Parse kriyakarama blocks to corpus
import json, re, uuid

# prefer OCR results if present; else fall back to page_texts
pages_combined = {}

# load page-level selectable text (if any)
if (OUT_DIR / "pdf_page_texts.json").exists():
    pt = json.loads((OUT_DIR / "pdf_page_texts.json").read_text(encoding="utf-8"))
    raw_texts = pt.get("page_texts", [])
else:
    raw_texts = []

ocr_file = OUT_DIR / "ocr_results_tesseract.json"
ocr_data = json.loads(ocr_file.read_text(encoding="utf-8")) if ocr_file.exists() else {}

n_pages = max(int(k) for k in list(ocr_data.keys()) + list(range(1, len(raw_texts)+1))) if raw_texts or ocr_data else 0

for p in range(1, n_pages+1):
    t = ""
    if p-1 < len(raw_texts) and raw_texts[p-1] and raw_texts[p-1].strip():
        t = raw_texts[p-1]
    elif str(p) in ocr_data and ocr_data[str(p)]:
        t = ocr_data[str(p)]
    pages_combined[str(p)] = t

heading_patterns = [r'ක්\s*රියා', r'ක්‍රියා', r'kriyakarama', r'ක්‍රියාකර', r'ක්‍රියාව']
heading_re = re.compile("|".join(heading_patterns), re.IGNORECASE)

corpus_items = []
manual_flags = []

for pnum_str, txt in pages_combined.items():
    pnum = int(pnum_str)
    if not txt or not txt.strip():
        continue
    if heading_re.search(txt):
        lines = [l.strip() for l in txt.splitlines() if l.strip()]
        section = None
        for ln in lines:
            if heading_re.search(ln):
                m = re.search(r'(\d{1,2})', ln)
                num = m.group(1) if m else "?"
                section = {"kriyakarama": f"kriyakarama{num}", "page": pnum, "lines": []}
                continue
            if section:
                section["lines"].append(ln)
        if section:
            for item in section["lines"]:
                rec = {
                    "id": str(uuid.uuid4()),
                    "kriyakarama": section["kriyakarama"],
                    "page": section["page"],
                    "original_sinhala": item,
                    "context": "sentence" if len(item.split())>1 else "word",
                    "notes": "extracted",
                    "singlish": None,
                    "phonemes": None
                }
                corpus_items.append(rec)
    else:
        if re.search(r'[\u0D80-\u0DFF]+', txt):
            manual_flags.append({"page": pnum, "reason": "Sinhala text present but no heading", "sample": txt[:200]})

# write outputs
with open(OUT_DIR / "corpus_kriyakarama.json", "w", encoding="utf-8") as f:
    json.dump(corpus_items, f, ensure_ascii=False, indent=2)
with open(OUT_DIR / "ocr_manual_flags.json", "w", encoding="utf-8") as f:
    json.dump(manual_flags, f, ensure_ascii=False, indent=2)

print("Wrote corpus_kriyakarama.json with", len(corpus_items), "items")
print("Wrote ocr_manual_flags.json with", len(manual_flags), "flags")
print("Files saved under:", OUT_DIR)


Wrote corpus_kriyakarama.json with 0 items
Wrote ocr_manual_flags.json with 0 flags
Files saved under: C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed


In [10]:
# Cell E — simple transliteration (stub, expand mapping as needed)
trans_map = {'අ':'a','ඉ':'i','ඊ':'ii','උ':'u','එ':'e','ඔ':'o','ක':'ka','ග':'ga','ච':'cha','ජ':'ja','ට':'ta','ඩ':'da','න':'na','ම':'ma','ය':'ya','ර':'ra','ල':'la','ව':'va','ස':'sa','හ':'ha'}
import json
corpus_path = OUT_DIR / "corpus_kriyakarama.json"
if corpus_path.exists():
    corpus = json.loads(corpus_path.read_text(encoding="utf-8"))
    def transliterate(s):
        return "".join(trans_map.get(ch, ch) for ch in s)
    for rec in corpus:
        rec["singlish"] = transliterate(rec["original_sinhala"])
    (OUT_DIR / "corpus_kriyakarama_translit.json").write_text(json.dumps(corpus, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wrote corpus_kriyakarama_translit.json")
else:
    print("corpus_kriyakarama.json not found. Run parsing cell first.")


Wrote corpus_kriyakarama_translit.json


In [11]:
# A: write combined raw text to out_sourcing_text.txt
from pathlib import Path
import json

OUT_DIR = Path(OUT_DIR)  # reuse existing variables
page_texts_file = OUT_DIR / "pdf_page_texts.json"
assert page_texts_file.exists(), "Run text extraction first."

data = json.loads(page_texts_file.read_text(encoding="utf-8"))
page_texts = data.get("page_texts", [])

combined_text = "\n\n".join([t for t in page_texts if t and t.strip()])
out_txt = OUT_DIR / "out_sourcing_text.txt"
out_txt.write_text(combined_text, encoding="utf-8")
print("Wrote:", out_txt)


Wrote: C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed\out_sourcing_text.txt


In [12]:
# B: OCR all pages and save token confidences (pytesseract image_to_data)
import fitz, io, json
from PIL import Image, ImageEnhance, ImageFilter
import pytesseract
from pathlib import Path

# optionally set tesseract.exe path on Windows:
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

PDF_PATH = Path(PDF_PATH)
OUT_DIR = Path(OUT_DIR)
doc = fitz.open(str(PDF_PATH))
n_pages = len(doc)

tess_lang = "sin+eng"
tess_config = "--psm 6"

def preprocess_pil(img):
    img = img.convert("L")
    img = ImageEnhance.Contrast(img).enhance(1.5)
    img = img.filter(ImageFilter.MedianFilter(size=3))
    return img

all_page_text = {}
all_page_data = {}   # token-level data for confidence checks

for pnum in range(1, n_pages+1):
    page = doc.load_page(pnum-1)
    txt = page.get_text("text") or ""
    if txt.strip():
        # save selectable text if present
        all_page_text[str(pnum)] = txt
        (OUT_DIR / f"page_{pnum}_text.txt").write_text(txt, encoding="utf-8")
        print(f"Page {pnum} selectable text saved ({len(txt)} chars)")
        continue

    pix = page.get_pixmap(dpi=300)
    pil_img = Image.open(io.BytesIO(pix.tobytes("png")))
    pil_img_proc = preprocess_pil(pil_img)

    # full page text
    try:
        page_text = pytesseract.image_to_string(pil_img_proc, lang=tess_lang, config=tess_config)
    except Exception:
        page_text = pytesseract.image_to_string(pil_img_proc, lang="eng", config=tess_config)
    page_text = page_text.replace("\r", "")
    all_page_text[str(pnum)] = page_text
    (OUT_DIR / f"ocr_page_{pnum}.txt").write_text(page_text, encoding="utf-8")

    # token-level data (DataFrame-like)
    data = pytesseract.image_to_data(pil_img_proc, lang=tess_lang, config=tess_config, output_type=pytesseract.Output.DICT)
    # data dict contains 'text' and 'conf'
    tokens = [{"text": t, "conf": c, "left": l, "top": top, "width": w, "height": h}
              for t, c, l, top, w, h in zip(data['text'], data['conf'], data['left'], data['top'], data['width'], data['height'])
              if t.strip()]
    all_page_data[str(pnum)] = tokens

    # save page image
    (OUT_DIR / f"page_{pnum}.png").write_bytes(pix.tobytes("png"))
    print(f"OCR page {pnum}: text {len(page_text)} chars; tokens {len(tokens)}")

# write aggregated outputs
(OUT_DIR / "ocr_all_pages_text.json").write_text(json.dumps(all_page_text, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT_DIR / "ocr_all_pages_tokens.json").write_text(json.dumps(all_page_data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved ocr_all_pages_text.json and ocr_all_pages_tokens.json")


Page 1 selectable text saved (275 chars)
OCR page 2: text 1016 chars; tokens 178
Page 3 selectable text saved (56 chars)
OCR page 4: text 912 chars; tokens 163
OCR page 5: text 496 chars; tokens 85
Page 6 selectable text saved (1472 chars)
Page 7 selectable text saved (64 chars)
Page 8 selectable text saved (545 chars)
Page 9 selectable text saved (243 chars)
Page 10 selectable text saved (100 chars)
Page 11 selectable text saved (677 chars)
Page 12 selectable text saved (872 chars)
Saved ocr_all_pages_text.json and ocr_all_pages_tokens.json


In [13]:
# C: extract sections by searching headings across full doc (multi-page)
import re, json, uuid
from pathlib import Path

OUT_DIR = Path(OUT_DIR)
ocr_text = json.loads((OUT_DIR / "ocr_all_pages_text.json").read_text(encoding="utf-8"))

# build a single ordered list of (page, line_no, line_text)
lines = []
for pstr in sorted(ocr_text.keys(), key=lambda x:int(x)):
    p = int(pstr)
    txt = ocr_text[pstr] or ""
    for ln_no, ln in enumerate(txt.splitlines(), start=1):
        if ln.strip():
            lines.append({"page": p, "ln_no": ln_no, "text": ln.strip()})

# heading regex: be permissive. include variants, allow small punctuation/whitespace changes
heading_patterns = [
    r'ක්\s*රියා', r'ක්‍රියා', r'ක්‍රියා\s*කරම', r'ක්‍රියා\s*කර', r'ක්‍රියාව',
    r'kriyakarama', r'kriya\s*karama', r'ක්‍රියාකරම', r'ක්‍රියාකරමා'
]
heading_re = re.compile("|".join(heading_patterns), re.IGNORECASE)

# find indices of heading lines
heading_indices = []
for i, rec in enumerate(lines):
    if heading_re.search(rec['text']):
        # try to extract a section number on same line
        m = re.search(r'(\d{1,2})', rec['text'])
        num = m.group(1) if m else None
        heading_indices.append((i, rec['page'], rec['text'], num))

# If none found, also try to find headings by searching for the Sinhala word 'ක්‍රියා' anywhere in the full page (alternative)
if not heading_indices:
    for i, rec in enumerate(lines):
        if 'ක්' in rec['text'] and 'රියා' in rec['text']:
            heading_indices.append((i, rec['page'], rec['text'], None))

print("Found heading occurrences:", len(heading_indices))
# build sections: from heading_i to heading_{i+1}-1
sections = []
for idx_i, (start_idx, start_page, start_text, num) in enumerate(heading_indices):
    end_idx = heading_indices[idx_i+1][0] if idx_i+1 < len(heading_indices) else len(lines)
    block_lines = [lines[j]['text'] for j in range(start_idx+1, end_idx)]
    sect_name = f"kriyakarama{num}" if num else f"kriyakarama_{idx_i+1}"
    sections.append({"kriyakarama": sect_name, "start_page": start_page, "lines": block_lines})

# Save sections for inspection
(OUT_DIR / "kriyakarama_sections_raw.json").write_text(json.dumps(sections, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved kriyakarama_sections_raw.json with", len(sections), "sections")


Found heading occurrences: 5
Saved kriyakarama_sections_raw.json with 5 sections


In [14]:
# D: normalize sections -> corpus.json
import json, uuid, re
from pathlib import Path

OUT_DIR = Path(OUT_DIR)
sections = json.loads((OUT_DIR / "kriyakarama_sections_raw.json").read_text(encoding="utf-8"))
corpus = []

# heuristics to split a section's lines into words or sentences:
# - if a line contains many short tokens separated by spaces or commas -> likely words
# - otherwise treat as sentence
for sec in sections:
    sec_name = sec.get("kriyakarama")
    for ln in sec.get("lines", []):
        # split on bullets, numbering, semicolons, commas
        parts = re.split(r'[\u2022•\-\–\—\)\(,;：:]+', ln)
        # if splitting gives many short items, treat each as an item
        items = []
        for p in parts:
            p = p.strip()
            if not p:
                continue
            # further split by space if tokens look like individual words (length 1-4)
            tokens = p.split()
            if len(tokens) > 1 and all(len(t) <= 4 for t in tokens[:3]):
                # ambiguous; keep whole line as one sentence
                items.append(p)
            else:
                # try splitting by whitespace for sequences of single glyph tokens (heuristic)
                # otherwise treat entire part as an item
                items.append(p)
        for it in items:
            rec = {
                "id": str(uuid.uuid4()),
                "kriyakarama": sec_name,
                "page_ref": sec.get("start_page"),
                "original_sinhala": it,
                "context": "sentence" if len(it.split()) > 1 else "word",
                "category": None,     # fill manually or with rules later
                "notes": "extracted_from_kriyakarama_sections",
                "singlish": None,
                "phonemes": None
            }
            corpus.append(rec)

# write corpus
(OUT_DIR / "corpus_kriyakarama_final.json").write_text(json.dumps(corpus, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote corpus_kriyakarama_final.json with", len(corpus), "records")


Wrote corpus_kriyakarama_final.json with 150 records


In [15]:
# E: produce manual QA file of low-confidence tokens
import json
from pathlib import Path

OUT_DIR = Path(OUT_DIR)
tokens_file = OUT_DIR / "ocr_all_pages_tokens.json"
if not tokens_file.exists():
    print("Run OCR-all (snippet B) first to get token confidences.")
else:
    data = json.loads(tokens_file.read_text(encoding="utf-8"))
    low_conf = []
    for pnum, toks in data.items():
        for t in toks:
            try:
                conf = float(t.get("conf", -1))
            except Exception:
                conf = -1
            if conf < 50:   # threshold (0-100). tune as needed.
                low_conf.append({"page": int(pnum), "text": t.get("text"), "conf": conf})
    (OUT_DIR / "ocr_low_confidence_tokens.json").write_text(json.dumps(low_conf, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wrote ocr_low_confidence_tokens.json with", len(low_conf), "low-confidence tokens")


Wrote ocr_low_confidence_tokens.json with 80 low-confidence tokens


In [16]:
# Sinhala -> Singlish transliteration with cluster/virama handling
# Paste into VS Code notebook and run.

import re

# Independent vowels (Sinhala characters) -> singlish
INDEPENDENT_VOWELS = {
    'අ': 'a',  'ආ': 'aa', 'ඇ': 'æ',  'ඈ': 'æa', 'ඉ': 'i',  'ඊ': 'ii',
    'උ': 'u',  'ඌ': 'uu', 'ඍ': 'ru', 'ඎ': 'ruu', 'එ': 'e',  'ඒ': 'ee',
    'ඔ': 'o',  'ඕ': 'oo', 'ඏ': 'i',  'ඐ': 'ii'
}

# Consonants base mapping
CONSONANTS = {
    'ක': 'k', 'ඛ': 'kh', 'ග': 'g', 'ඝ': 'gh', 'ඞ': 'ng',
    'ච': 'ch','ඡ': 'chh','ජ': 'j', 'ඣ': 'jh', 'ඤ': 'ny',
    'ට': 't', 'ඨ': 'th', 'ඩ': 'd', 'ඪ': 'dh', 'ණ': 'n',
    'ත': 't', 'ථ': 'th', 'ද': 'd', 'ධ': 'dh', 'න': 'n',
    'ප': 'p', 'ඵ': 'ph', 'බ': 'b', 'භ': 'bh', 'ම': 'm',
    'ය': 'y', 'ර': 'r', 'ල': 'l', 'ව': 'v', 
    'ශ': 'sh','ෂ': 'sh', 'ස': 's', 'හ': 'h', 'ළ': 'l',
    'ෆ': 'f', 'ඥ': 'gn', 'ඛ': 'kh', 'ඨ': 'th'
}

# Dependent vowel signs (attached to consonant) -> singlish
# Using the common Sinhala vowel signs characters
VOWEL_SIGNS = {
    'ා': 'aa',   # U+0DCF
    'ි': 'i',    # U+0DD2
    'ී': 'ii',   # U+0DD3
    'ු': 'u',    # U+0DD4
    'ූ': 'uu',   # U+0DD6
    'ෙ': 'e',    # U+0DD9
    'ේ': 'ee',   # U+0DDA
    'ෛ': 'ai',   # U+0DDB
    'ො': 'o',    # U+0DDB or similar (use 'ො')
    'ෝ': 'oo',  # long o sign (දෝ) - often represented as two chars, handle heuristically
    'ෞ': 'au',   # U+0DDD
    'ෟ': 'e',    # rare
}

# Virama (hal kirīma) that suppresses inherent 'a' (U+0DCA)
VIRAMA = '්'

# Anusvara and visarga
ANUSVARA = 'ං'
VISARGA = 'ඃ'
SPECIAL_MARKS = {ANUSVARA: 'n', VISARGA: 'h'}

# Some punctuation that may appear; keep as space or remove
PUNCT = set('.,;:?!\"\'()[]{}-–—·•/\\|')

def transliterate_sinhala(text: str) -> str:
    """
    Transliterate Sinhala to Singlish (Latin) with simple cluster handling.
    - Consonant + VIRAMA + consonant -> combine base consonants without inherent 'a' (cluster).
    - Consonant + vowel sign -> consonant_map + vowel_map.
    - Consonant alone -> consonant_map + 'a' (inherent vowel).
    - Independent vowels map directly.
    - Special marks (anusvara/visarga) handled.
    - Unknown characters: digits/latin kept; other unknowns skipped.
    """
    out = []
    i = 0
    n = len(text)
    while i < n:
        ch = text[i]

        # independent vowel
        if ch in INDEPENDENT_VOWELS:
            out.append(INDEPENDENT_VOWELS[ch])
            i += 1
            continue

        # special marks
        if ch in SPECIAL_MARKS:
            out.append(SPECIAL_MARKS[ch])
            i += 1
            continue

        # whitespace/punctuation
        if ch.isspace():
            out.append(' ')
            i += 1
            continue
        if ch in PUNCT:
            # keep punctuation as space to separate tokens
            out.append(' ')
            i += 1
            continue

        # consonant - handle cluster and vowel signs
        if ch in CONSONANTS:
            base = CONSONANTS[ch]

            # lookahead
            next_i = i + 1
            # cluster: consonant + VIRAMA + consonant
            if next_i < n and text[next_i] == VIRAMA:
                # skip virama and combine with next consonant if exists
                next_next = next_i + 1
                if next_next < n and text[next_next] in CONSONANTS:
                    next_base = CONSONANTS[text[next_next]]
                    # produce cluster like 'k' + next_base (e.g., kr, ksh)
                    out.append(base + next_base)
                    i = next_next + 1
                    continue
                else:
                    # lone virama (unexpected) -> emit base without vowel
                    out.append(base)
                    i = next_i + 1
                    continue

            # if next char is a vowel sign, attach vowel
            if next_i < n and text[next_i] in VOWEL_SIGNS:
                vowel = VOWEL_SIGNS[text[next_i]]
                out.append(base + vowel)
                i = next_i + 1
                continue

            # some texts may encode long 'o' or compound vowel with two chars
            # check two-char vowel combos (e.g., 'ො' + 'ා' etc.) — rudimentary handling
            if next_i < n:
                two = text[next_i:next_i+2]
                # handle 'ෝ' or similar sequences heuristically
                if two and two in VOWEL_SIGNS:
                    out.append(base + VOWEL_SIGNS[two])
                    i = next_i + 2
                    continue

            # no vowel sign, no virama -> inherent 'a'
            out.append(base + 'a')
            i += 1
            continue

        # vowel sign appearing alone (rare) -> map if known
        if ch in VOWEL_SIGNS:
            out.append(VOWEL_SIGNS[ch])
            i += 1
            continue

        # digits / ascii letters
        if re.match(r'[A-Za-z0-9]', ch):
            out.append(ch)
            i += 1
            continue

        # unknown char: try to keep basic punctuation, else skip
        # optionally append it (skip to avoid garbage)
        i += 1

    # finalize: collapse multiple spaces
    s = ''.join(out)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# quick internal tests to verify behavior
_examples = ["ගස", "කිරි", "මම", "සිංහල", "ක්‍රියාව 03", "ආයුබෝවන්"]
for ex in _examples:
    print(f"{ex} -> {transliterate_sinhala(ex)}")


ගස -> gasa
කිරි -> kiri
මම -> mama
සිංහල -> sinhala
ක්‍රියාව 03 -> kriyaava 03
ආයුබෝවන් -> aayuboovan


In [17]:
# Integration cell: add 'singlish' to corpus_kriyakarama_final.json
import json
from pathlib import Path

OUT_DIR = Path(r"C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed")
in_file = OUT_DIR / "corpus_kriyakarama_final.json"
out_file = OUT_DIR / "corpus_kriyakarama_final_translit.json"

if not in_file.exists():
    raise FileNotFoundError(f"{in_file} not found. Run the extraction/normalization first.")

corpus = json.loads(in_file.read_text(encoding="utf-8"))

# import transliterator function from current notebook (assumes it's in the same namespace)
# if you put transliterate_sinhala in a different module, import accordingly.
for rec in corpus:
    original = rec.get("original_sinhala", "") or ""
    rec["singlish"] = transliterate_sinhala(original)

out_file.write_text(json.dumps(corpus, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {len(corpus)} records with 'singlish' to {out_file}")
# show 10 sample entries
for r in corpus[:10]:
    print(r.get("original_sinhala"), "->", r.get("singlish"))


Wrote 150 records with 'singlish' to C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed\corpus_kriyakarama_final_translit.json
DY] ඔහු ගසේ ඔල / මුල කැඩුවා. -> DY ohu gasee ola mula kaduvaa
| නංගී කජු / කේජු කන්න ආසයි -> nangii kaju keeju kann aasayi
1 | ගෝනියේ කාල්‌ / හිල්‌ තියෙනවා. -> 1 gooniyee kaal hil tiyenavaa
| ගිය මාසේ කඩපු තැඹිලි ටික පල්‌ / පෞල්‌ වෙලා. -> giya maasee kadapu taili tika pal paul velaa
| මල්ලී බුූල්ලාව / බිල්ලාව දැක්කද? -> mallii buuullaava billaava dakkda
ස්වර ශබ්ද හඳුනා ගැනිම -> svra shabd haunaa ganima
කෙකේද"” -> kekeeda
චිකින්සකචරයං සෑම පේලියකම QF සෑම චචනයක්ම -> chikinskacharayan sama peeliyakama QF sama chachanayakm
ඇඟිල්ල 283s! ශබ්ඳ නගා කියවයි. චිකින්සකවරයා -> æill 283s shab nagaa kiyavayi chikinskavarayaa
ඉන්‌ පසු එක්‌ චචනයක්‌ තෝ රාගෙන කියවු විට ඔබ එම චචනය හඳුනා ගත යුතුය. අකා පමණක්‌ wer ගෑනිම -> in pasu e

In [18]:
# Advanced cluster transliteration helper (optional)
def transliterate_sinhala_advanced(text: str) -> str:
    # Similar to transliterate_sinhala above but when encountering C + VIRAMA + C2,
    # create cluster = base(C) + base(C2) and then check if C2 has a vowel sign following to decide to add 'a'.
    out = []
    i = 0
    n = len(text)
    while i < n:
        ch = text[i]
        if ch in CONSONANTS:
            base = CONSONANTS[ch]
            next_i = i+1
            if next_i < n and text[next_i] == VIRAMA:
                # lookahead to next consonant
                nn = next_i + 1
                if nn < n and text[nn] in CONSONANTS:
                    next_base = CONSONANTS[text[nn]]
                    # check whether the consonant after next has vowel sign -> decide trailing vowel
                    after_nn = nn + 1
                    if after_nn < n and text[after_nn] in VOWEL_SIGNS:
                        # cluster + vowel of next consonant
                        vowel = VOWEL_SIGNS[text[after_nn]]
                        out.append(base + next_base + vowel)
                        i = after_nn + 1
                        continue
                    else:
                        # cluster without an explicit following vowel -> keep cluster (no 'a') and advance
                        out.append(base + next_base)
                        i = nn + 1
                        continue
            # fallback to normal rules
            if next_i < n and text[next_i] in VOWEL_SIGNS:
                out.append(base + VOWEL_SIGNS[text[next_i]])
                i = next_i + 1
                continue
            out.append(base + 'a')
            i += 1
            continue
        # other handlers (as earlier)...
        if ch in INDEPENDENT_VOWELS:
            out.append(INDEPENDENT_VOWELS[ch]); i+=1; continue
        if ch in SPECIAL_MARKS:
            out.append(SPECIAL_MARKS[ch]); i+=1; continue
        if ch.isspace():
            out.append(' '); i+=1; continue
        i += 1
    s = ''.join(out); s = re.sub(r'\s+', ' ', s).strip()
    return s


In [19]:
# quick spot-check of transliterated corpus
import json, random
from pathlib import Path
OUT_DIR = Path(r"C:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\data\processed")
docs = json.loads((OUT_DIR / "corpus_kriyakarama_final_translit.json").read_text(encoding="utf-8"))
for rec in random.sample(docs, min(20, len(docs))):
    print(rec["kriyakarama"], rec["page_ref"], rec["original_sinhala"], "->", rec["singlish"])


kriyakarama13 5 Activity1 -> Activity1
kriyakarama13 5 Summarize -> Summarize
kriyakarama13 5 BETTER IF YOU CAN IDENTIFY PATIENTS EXACT FREQUENCY CATEGORY THROUGH ABOVE -> BETTER IF YOU CAN IDENTIFY PATIENTS EXACT FREQUENCY CATEGORY THROUGH ABOVE
kriyakarama13 5 SENTENCE LEVEL IDENTIFICATION -> SENTENCE LEVEL IDENTIFICATION
kriyakarama13 5 for emergency related activities at the end emergency comprehension -> for emergency related activities at the end emergency comprehension
kriyakarama11 5 විකින්සංක වරය්‍ය eS ed@ond ආති සැම වචනයක්ප -> vikinsnka varayya eS edond aati sama vachanayakp
kriyakarama13 5 For each user to do GAM analysis performance graph needed to be drawn for each activity -> For each user to do GAM analysis performance graph needed to be drawn for each activity
kriyakarama03 2 ස්වර ශබ්ද හඳුනා ගැනිම -> svra shabd haunaa ganima
kriyakarama13 5 IF THE USER IS ABLE IDENTIFY 2 WORDS SENTENCES -> IF THE USER IS ABLE IDENTIFY 2 WORDS SENTENCES
kriyakarama04 2 omens? -> omens
kr

step 4

In [20]:
%pip install pymongo pydub soundfile librosa numpy scipy pyttsx3 gTTS



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
%pip install soundfile


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# Cell 1 — config (adjust if your notebook lives elsewhere)
from pathlib import Path
import json, os

# Try to infer project root: go up until you find a folder named 'Models' or use cwd parent
CWD = Path.cwd()
ROOT = CWD
for _ in range(4):
    if (ROOT / "Models").exists() or (ROOT / "data").exists() or (ROOT / "Data").exists():
        break
    ROOT = ROOT.parent

# Common data paths (case-insensitive handling)
DATA_DIR = ROOT / "Data" if (ROOT / "Data").exists() else ROOT / "data" if (ROOT / "data").exists() else ROOT / "Models" / "data"
PROCESSED = DATA_DIR / "processed"
AUDIO_DIR = DATA_DIR / "audio"
MODELS_DIR = ROOT / "models"
SCRIPTS_DIR = ROOT / "data" / "scripts" if (ROOT / "data" / "scripts").exists() else ROOT / "Data" / "scripts"
DOCS_DIR = ROOT / "docs"
PROCESSED.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROCESSED:", PROCESSED)
print("AUDIO_DIR:", AUDIO_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("SCRIPTS_DIR:", SCRIPTS_DIR)
print("DOCS_DIR:", DOCS_DIR)


ROOT: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models
DATA_DIR: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\Data
PROCESSED: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\Data\processed
AUDIO_DIR: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\Data\audio
MODELS_DIR: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehens

In [23]:
# Cell 2 — transliterate_sinhala function (run once)
import re

# Comprehensive-ish mapping; extend if you see missing characters.
INDEPENDENT_VOWELS = {
    'අ':'a','ආ':'aa','ඇ':'ae','ඈ':'aae','ඉ':'i','ඊ':'ii','උ':'u','ඌ':'uu',
    'එ':'e','ඒ':'ee','ඔ':'o','ඕ':'oo','ඍ':'ru','ඎ':'ruu'
}
CONSONANTS = {
 'ක':'k','ඛ':'kh','ග':'g','ඝ':'gh','ඞ':'ng','ච':'ch','ඡ':'chh','ජ':'j','ඣ':'jh','ඤ':'ny',
 'ට':'t','ඨ':'th','ඩ':'d','ඪ':'dh','ණ':'n','ත':'t','ථ':'th','ද':'d','ධ':'dh','න':'n',
 'ප':'p','ඵ':'ph','බ':'b','භ':'bh','ම':'m','ය':'y','ර':'r','ල':'l','ව':'v','ශ':'sh','ෂ':'sh',
 'ස':'s','හ':'h','ළ':'l','ෆ':'f','ඥ':'gn'
}
VOWEL_SIGNS = {
 'ා':'aa','ਿ':'i','ී':'ii','ු':'u','ූ':'uu','ෙ':'e','ේ':'ee','ෛ':'ai','ො':'o','ෝ':'oo','ෞ':'au'
}
VIRAMA = '්'
ANUSVARA = 'ං'
VISARGA = 'ඃ'
SPECIAL = {ANUSVARA:'n', VISARGA:'h'}
PUNCT = set('.,;:?!\"\'()[]{}-–—·•/\\|')

def transliterate_sinhala(text: str) -> str:
    out = []
    i = 0
    n = len(text)
    while i < n:
        ch = text[i]
        # independent vowels
        if ch in INDEPENDENT_VOWELS:
            out.append(INDEPENDENT_VOWELS[ch]); i += 1; continue
        if ch in SPECIAL:
            out.append(SPECIAL[ch]); i += 1; continue
        if ch.isspace():
            out.append(' '); i += 1; continue
        if ch in PUNCT:
            out.append(' '); i += 1; continue
        # consonant handling
        if ch in CONSONANTS:
            base = CONSONANTS[ch]
            # cluster: C + VIRAMA + C2
            if i+1 < n and text[i+1] == VIRAMA:
                if i+2 < n and text[i+2] in CONSONANTS:
                    next_base = CONSONANTS[text[i+2]]
                    # look for vowel sign after C2
                    if i+3 < n and text[i+3] in VOWEL_SIGNS:
                        vowel = VOWEL_SIGNS[text[i+3]]
                        out.append(base + next_base + vowel)
                        i = i+4; continue
                    else:
                        out.append(base + next_base)
                        i = i+3; continue
                else:
                    out.append(base); i += 2; continue
            # vowel sign after consonant?
            if i+1 < n and text[i+1] in VOWEL_SIGNS:
                out.append(base + VOWEL_SIGNS[text[i+1]]); i += 2; continue
            # default inherent vowel 'a'
            out.append(base + 'a'); i += 1; continue
        # vowel sign alone
        if ch in VOWEL_SIGNS:
            out.append(VOWEL_SIGNS[ch]); i += 1; continue
        # digits/latin
        if re.match(r'[A-Za-z0-9]', ch):
            out.append(ch); i += 1; continue
        # unknown char -> skip
        i += 1
    s = ''.join(out)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# quick smoke test
tests = ["ගස","කිරි","මම","සිංහල","ක්‍රියාව 03","ආයුබෝවන්"]
for t in tests:
    print(t, "->", transliterate_sinhala(t))


ගස -> gasa
කිරි -> kara
මම -> mama
සිංහල -> sanhala
ක්‍රියාව 03 -> krayaava 03
ආයුබෝවන් -> aayuboovan


In [24]:
# Cell 3 — transliterate corpus file and save corpus_transliterated.json
from pathlib import Path
import json
IN = PROCESSED / "corpus_kriyakarama_final.json"
OUT = PROCESSED / "corpus_transliterated.json"

if not IN.exists():
    print("Missing", IN, "- ensure corpus_kriyakarama_final.json exists (run Step3 parsing first).")
else:
    data = json.loads(IN.read_text(encoding="utf-8"))
    for rec in data:
        text = rec.get("original_sinhala","") or ""
        rec["singlish"] = transliterate_sinhala(text)
    OUT.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wrote", OUT, "with", len(data), "records")


Wrote c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\Data\processed\corpus_transliterated.json with 150 records


In [25]:
# Cell 5 — create/save a phoneme_map.json skeleton (edit to refine)
import json
phonemap = {
  "a":["a"], "aa":["a:"], "i":["i"], "ii":["i:"], "u":["u"], "uu":["u:"],
  "e":["e"], "ee":["e:"], "o":["o"], "oo":["o:"], "ai":["ai"], "au":["au"],
  "k":["k"], "kh":["kʰ"], "g":["g"], "ng":["ŋ"], "ch":["tʃ"], "j":["dʒ"],
  "t":["t"], "d":["d"], "n":["n"], "p":["p"], "b":["b"], "m":["m"],
  "y":["j"], "r":["r"], "l":["l"], "v":["v"], "s":["s"], "sh":["ʃ"], "h":["h"], "f":["f"]
}
(MODELS_DIR / "phoneme_map.json").write_text(json.dumps(phonemap, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote models/phoneme_map.json with", len(phonemap), "entries")


Wrote models/phoneme_map.json with 32 entries


In [26]:
# Cell 6 — build lexicon.json and phoneme_index.json; seed Mongo if available
import json, re
from pathlib import Path
from pymongo import MongoClient

IN = PROCESSED / "corpus_transliterated.json"
PHONEME_FILE = MODELS_DIR / "phoneme_map.json"
OUT_LEX = MODELS_DIR / "lexicon.json"
OUT_INDEX = MODELS_DIR / "phoneme_index.json"

if not IN.exists():
    print("Missing transliterated corpus:", IN)
else:
    docs = json.loads(IN.read_text(encoding="utf-8"))
    phonemap = json.loads(PHONEME_FILE.read_text(encoding="utf-8"))
    keys = sorted(phonemap.keys(), key=lambda x: -len(x))  # longest-first

    def tokenize_singlish(word):
        w = (word or "").strip().lower()
        tokens = []
        i = 0
        while i < len(w):
            matched = False
            for k in keys:
                if w.startswith(k, i):
                    tokens.append(k); i += len(k); matched = True; break
            if not matched:
                tokens.append(w[i]); i += 1
        return tokens

    lexicon = []
    phoneme_index = {}
    for rec in docs:
        wid = rec.get("id")
        sing = (rec.get("singlish") or "").strip().lower()
        letters = tokenize_singlish(sing)
        phonemes = []
        for t in letters:
            if t in phonemap:
                phonemes.extend(phonemap[t])
            else:
                phonemes.append(t)
        target_letters = letters.copy()
        entry = {
            "word_id": wid, "sinhala": rec.get("original_sinhala"), "singlish": sing,
            "letters": letters, "phonemes": phonemes, "target_letters": target_letters,
            "category": rec.get("category"), "kriyakarama": rec.get("kriyakarama"), "page_ref": rec.get("page_ref")
        }
        lexicon.append(entry)
        for p in set(phonemes):
            phoneme_index.setdefault(p, []).append(wid)

    OUT_LEX.write_text(json.dumps(lexicon, ensure_ascii=False, indent=2), encoding="utf-8")
    OUT_INDEX.write_text(json.dumps(phoneme_index, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wrote", OUT_LEX, "(", len(lexicon), "entries )")
    print("Wrote", OUT_INDEX, "(", len(phoneme_index), "phonemes )")

    # Optional Mongo seed (uncomment to use)
    try:
        client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=2000)
        client.server_info()  # check connection
        db = client["farm_auditory"]
        coll = db["lexicon"]
        # coll.drop()   # uncomment to reset
        if lexicon:
            coll.insert_many(lexicon)
            print("Seeded Mongo DB farm_auditory.lexicon with", len(lexicon), "docs")
    except Exception as e:
        print("Mongo not available or connection failed:", e)


Wrote c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\models\lexicon.json ( 150 entries )
Wrote c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\models\phoneme_index.json ( 45 phonemes )
Seeded Mongo DB farm_auditory.lexicon with 150 docs


In [27]:
# Cell 7 — quick lookup for examples of a phoneme (e.g., 'k' or 'kʰ')
import json
PI = MODELS_DIR / "phoneme_index.json"
LEX = MODELS_DIR / "lexicon.json"
if PI.exists() and LEX.exists():
    phon_idx = json.loads(PI.read_text(encoding="utf-8"))
    lex = json.loads(LEX.read_text(encoding="utf-8"))
    sample_ph = "k"
    ids = phon_idx.get(sample_ph, [])[:10]
    examples = [next((x for x in lex if x["word_id"]==i), None) for i in ids]
    print("Examples for phoneme", sample_ph, "->")
    for ex in examples:
        if ex:
            print(ex["sinhala"], ex["singlish"], ex["phonemes"])
else:
    print("Run lexicon build first (cell 6).")


Examples for phoneme k ->
DY] ඔහු ගසේ ඔල / මුල කැඩුවා. dy ohu gasee ola mula kaduvaa ['d', 'j', ' ', 'o', 'h', 'u', ' ', 'g', 'a', 's', 'e:', ' ', 'o', 'l', 'a', ' ', 'm', 'u', 'l', 'a', ' ', 'k', 'a', 'd', 'u', 'v', 'a:']
| නංගී කජු / කේජු කන්න ආසයි nangii kaju keeju kann aasaya ['n', 'a', 'ŋ', 'i:', ' ', 'k', 'a', 'dʒ', 'u', ' ', 'k', 'e:', 'dʒ', 'u', ' ', 'k', 'a', 'n', 'n', ' ', 'a:', 's', 'a', 'j', 'a']
1 | ගෝනියේ කාල්‌ / හිල්‌ තියෙනවා. 1 goonayee kaal hal tayenavaa ['1', ' ', 'g', 'o:', 'n', 'a', 'j', 'e:', ' ', 'k', 'a:', 'l', ' ', 'h', 'a', 'l', ' ', 't', 'a', 'j', 'e', 'n', 'a', 'v', 'a:']
| ගිය මාසේ කඩපු තැඹිලි ටික පල්‌ / පෞල්‌ වෙලා. gaya maasee kadapu tala taka pal paul velaa ['g', 'a', 'j', 'a', ' ', 'm', 'a:', 's', 'e:', ' ', 'k', 'a', 'd', 'a', 'p', 'u', ' ', 't', 'a', 'l', 'a', ' ', 't', 'a', 'k', 'a', ' ', 'p', 'a', 'l', ' ', 'p', 'au', 'l', ' ', 'v', 'e', 'l', 'a:']
| මල්ලී බුූල්ලාව / බිල්ලාව දැක්කද? mallii buuullaava ballaava dakkda ['m', 'a', 'l', 'l', 'i:', ' ', 'b'

In [28]:
# Cell 8 — write docs/transliteration.md and docs/audio_decision.md skeletons
translit_md = DOCS_DIR / "transliteration.md"
translit_md.write_text(
"# Transliteration choices\n\n"
"- Approach: rule-based mapping (deterministic) implemented in notebook.\n"
"- Rationale: consistent mapping for downstream NLP and frontend display.\n"
"- Notes: mapping stored at models/phoneme_map.json; transliteration function in notebook.\n", encoding="utf-8")
audio_md = DOCS_DIR / "audio_decision.md"
audio_md.write_text(
"# Audio decision log\n\n"
"- Preferred: Human-recorded native speaker audio stored under Data/audio/words/ (recommended).\n"
"- Fallback: gTTS-generated audio for quick prototypes (quality may vary).\n", encoding="utf-8")
print("Wrote docs:", translit_md, "and", audio_md)


Wrote docs: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\docs\transliteration.md and c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\docs\audio_decision.md


In [29]:
# Cell 9 — generate TTS MP3s for first N words (gTTS, requires internet)
from gtts import gTTS
import json, time
from pathlib import Path

IN = PROCESSED / "corpus_transliterated.json"
OUT = AUDIO_DIR / "words_tts"
OUT.mkdir(parents=True, exist_ok=True)

if not IN.exists():
    print("Missing transliterated corpus:", IN)
else:
    docs = json.loads(IN.read_text(encoding="utf-8"))
    N = min(50, len(docs))  # change batch size if you want
    count = 0
    for rec in docs[:N]:
        wid = rec.get("id")
        text = rec.get("original_sinhala") or rec.get("singlish") or ""
        if not text:
            continue
        try:
            tts = gTTS(text=text, lang='si')  # try Sinhala; if fails use 'en'
            outp = OUT / f"{wid}.mp3"
            tts.save(str(outp))
            count += 1
            print("Saved TTS:", outp.name)
        except Exception as e:
            print("TTS failed for", wid, e)
        time.sleep(0.4)
    print("Generated", count, "TTS files in", OUT)


Saved TTS: ff275abd-2ae3-4dc8-b358-211b759bfd92.mp3
Saved TTS: 2791972b-999e-4663-81a2-5ba5b9fd4bde.mp3
Saved TTS: de21de3d-fb34-4267-b7d3-9b0e5e3212d6.mp3
Saved TTS: e2e04eb6-2ffa-4860-8151-af524d840b2c.mp3
Saved TTS: 0fe7a5c2-fe23-46ef-9c93-5fb8da387a9b.mp3
Saved TTS: 668f41a2-1d32-4d85-83f7-5724b26913ee.mp3
Saved TTS: b3977999-cc01-4539-b700-c0ab1c0bae5d.mp3
Saved TTS: 57066a97-d034-4508-af16-6fac51f24b60.mp3
Saved TTS: 1630b845-8673-43bc-8b6a-4c0dec605b5f.mp3
Saved TTS: 895358be-0c2f-4b8f-a4eb-6b801fb2e653.mp3
Saved TTS: 7aba6775-2faa-4495-aaa3-6a68256c5d24.mp3
Saved TTS: cc61c821-c096-4f6a-b2d9-1ee8cd5f7c21.mp3
Saved TTS: bf3035ef-7147-4a18-89df-172fdca6cbf0.mp3
Saved TTS: ec71f851-7141-43df-a3d2-a7c384777fe2.mp3
Saved TTS: 04794cea-76c7-4c13-9321-2ade308429d8.mp3
Saved TTS: 4f66bb94-a343-4dc3-97dc-6df8b06bcf37.mp3
Saved TTS: 2e4349ea-26dc-4c27-9c60-d7d3a3a5d9a9.mp3
Saved TTS: caca6f4e-0975-4729-9212-057bde3c7259.mp3
Saved TTS: 503e497f-db47-4a72-81d7-bd57ee217447.mp3
Saved TTS: 0

In [30]:
# Check ffmpeg/ffprobe availability
import shutil, sys
ffmpeg_path = shutil.which("ffmpeg")
ffprobe_path = shutil.which("ffprobe")
print("ffmpeg on PATH:", bool(ffmpeg_path))
print("ffmpeg path:", ffmpeg_path)
print("ffprobe on PATH:", bool(ffprobe_path))
print("Python:", sys.version)


ffmpeg on PATH: True
ffmpeg path: C:\Users\94772\Downloads\ffmpeg-8.0-essentials_build\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE
ffprobe on PATH: True
Python: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]


In [34]:
import shutil
print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))


ffmpeg: C:\Users\94772\Downloads\ffmpeg-8.0-essentials_build\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE
ffprobe: C:\Users\94772\Downloads\ffmpeg-8.0-essentials_build\ffmpeg-8.0-essentials_build\bin\ffprobe.EXE


In [31]:
# Cell 10 — normalize audio files (mp3 -> 16kHz mono wav) using pydub
from pydub import AudioSegment
from pathlib import Path

IN_DIR = AUDIO_DIR / "words_tts"
OUT_DIR = AUDIO_DIR / "words_norm"
IN_DIR = IN_DIR if IN_DIR.exists() else AUDIO_DIR  # fallback
OUT_DIR.mkdir(parents=True, exist_ok=True)

for f in IN_DIR.glob("*.*"):
    try:
        audio = AudioSegment.from_file(f)
        audio = audio.set_channels(1)
        audio = audio.set_frame_rate(16000)
        outp = OUT_DIR / (f.stem + ".wav")
        audio.export(outp, format="wav")
        print("Normalized", f.name, "->", outp.name)
    except Exception as e:
        print("Failed to normalize", f, e)
print("Normalization step complete; check", OUT_DIR)


Normalized 00ff8d4e-af63-443f-a8ef-30d6afc3b7c0.mp3 -> 00ff8d4e-af63-443f-a8ef-30d6afc3b7c0.wav
Normalized 02714415-6eda-484b-9504-9032290b015d.mp3 -> 02714415-6eda-484b-9504-9032290b015d.wav
Normalized 03bc684c-72d5-4404-b96f-14204596c080.mp3 -> 03bc684c-72d5-4404-b96f-14204596c080.wav
Normalized 03bff4d8-7243-48e5-845f-2999ed46ab33.mp3 -> 03bff4d8-7243-48e5-845f-2999ed46ab33.wav
Normalized 045c8098-9948-479c-a043-8dad4fb77682.mp3 -> 045c8098-9948-479c-a043-8dad4fb77682.wav
Normalized 04794cea-76c7-4c13-9321-2ade308429d8.mp3 -> 04794cea-76c7-4c13-9321-2ade308429d8.wav
Normalized 04b7e879-8bd0-449c-93d7-4820203f5394.mp3 -> 04b7e879-8bd0-449c-93d7-4820203f5394.wav
Normalized 08f1c870-d4aa-4a31-9c5b-f5b9d60eccc6.mp3 -> 08f1c870-d4aa-4a31-9c5b-f5b9d60eccc6.wav
Normalized 0ae2ea06-bf63-4bf6-a2ae-396f4de8719d.mp3 -> 0ae2ea06-bf63-4bf6-a2ae-396f4de8719d.wav
Normalized 0ec45ab1-22aa-46e2-b524-f7fe73d52532.mp3 -> 0ec45ab1-22aa-46e2-b524-f7fe73d52532.wav
Normalized 0fe7a5c2-fe23-46ef-9c93-5fb8d

In [32]:
# Cell 11 — write backend route file for audio streaming (save to backend/app/api/v1/routes_audio.py)
from pathlib import Path
BACKEND_DIR = ROOT / "backend" / "app" / "api" / "v1"
BACKEND_DIR.mkdir(parents=True, exist_ok=True)
ROUTE_FILE = BACKEND_DIR / "routes_audio.py"

route_code = f'''from fastapi import APIRouter, HTTPException
from fastapi.responses import FileResponse
from pathlib import Path

router = APIRouter(prefix="/api/v1/audio", tags=["audio"])
AUDIO_ROOT = Path(r"{(AUDIO_DIR / 'words_norm').as_posix()}")

@router.get("/word/{{word_id}}")
def get_word_audio(word_id: str):
    p = AUDIO_ROOT / f"{{word_id}}.wav"
    if not p.exists():
        raise HTTPException(status_code=404, detail="Audio not found")
    return FileResponse(str(p), media_type="audio/wav", filename=f"{{word_id}}.wav")
'''
ROUTE_FILE.write_text(route_code, encoding="utf-8")
print("Wrote FastAPI route to", ROUTE_FILE)
print("Remember to include this router in your main FastAPI app (include_router).")


Wrote FastAPI route to c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\backend\app\api\v1\routes_audio.py
Remember to include this router in your main FastAPI app (include_router).


In [33]:
# Cell 12 — print minimal React AudioPlayer component for frontend (save as src/components/AudioPlayer.jsx)
jsx = r'''
import React, { useRef, useState, useEffect } from "react";

export default function AudioPlayer({ src }) {
  const audioRef = useRef(null);
  const [playing, setPlaying] = useState(false);
  const [progress, setProgress] = useState(0);

  useEffect(() => {
    const audio = audioRef.current;
    if (!audio) return;
    const onTime = () => setProgress((audio.currentTime / (audio.duration || 1)) || 0);
    const onEnd = () => setPlaying(false);
    audio.addEventListener("timeupdate", onTime);
    audio.addEventListener("ended", onEnd);
    return () => {
      audio.removeEventListener("timeupdate", onTime);
      audio.removeEventListener("ended", onEnd);
    };
  }, []);

  useEffect(() => {
    if (!audioRef.current) return;
    if (playing) audioRef.current.play();
    else audioRef.current.pause();
  }, [playing]);

  return (
    <div className="audio-player">
      <audio ref={audioRef} src={src} preload="auto" />
      <button onClick={() => setPlaying(!playing)}>{playing ? "Pause" : "Play"}</button>
      <div style={{ width: "200px", height: "8px", background: "#eee", display: "inline-block", marginLeft: 8 }}>
        <div style={{ width: `${Math.round(progress * 100)}%`, height: "100%", background: "#4caf50" }} />
      </div>
      <button onClick={() => { if (audioRef.current) audioRef.current.currentTime = 0; }}>Restart</button>
      <button onClick={() => { /* trigger Other flow in your UI */ }}>I didn't hear any</button>
    </div>
  );
}
'''
print(jsx)
# Optionally write to file (uncomment if you want file written)
# (ROOT / "frontend" / "src" / "components").mkdir(parents=True, exist_ok=True)
# (ROOT / "frontend" / "src" / "components" / "AudioPlayer.jsx").write_text(jsx, encoding="utf-8")



import React, { useRef, useState, useEffect } from "react";

export default function AudioPlayer({ src }) {
  const audioRef = useRef(null);
  const [playing, setPlaying] = useState(false);
  const [progress, setProgress] = useState(0);

  useEffect(() => {
    const audio = audioRef.current;
    if (!audio) return;
    const onTime = () => setProgress((audio.currentTime / (audio.duration || 1)) || 0);
    const onEnd = () => setPlaying(false);
    audio.addEventListener("timeupdate", onTime);
    audio.addEventListener("ended", onEnd);
    return () => {
      audio.removeEventListener("timeupdate", onTime);
      audio.removeEventListener("ended", onEnd);
    };
  }, []);

  useEffect(() => {
    if (!audioRef.current) return;
    if (playing) audioRef.current.play();
    else audioRef.current.pause();
  }, [playing]);

  return (
    <div className="audio-player">
      <audio ref={audioRef} src={src} preload="auto" />
      <button onClick={() => setPlaying(!playing)}>{playing ? "

In [35]:
# Cell 13 — create audio manifest skeleton CSV for human recordings
import csv, json
from pathlib import Path
OUT_CSV = AUDIO_DIR / "audio_manifest.csv"
CORP = PROCESSED / "corpus_kriyakarama_final.json"
rows = []
if CORP.exists():
    docs = json.loads(CORP.read_text(encoding="utf-8"))
    with OUT_CSV.open("w", newline='', encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id","type","path","recorded_by","date","notes"])
        for d in docs:
            w.writerow([d["id"], "word", str((AUDIO_DIR / "words" / f"{d['id']}.wav").as_posix()), "", "", ""])
    print("Created audio manifest:", OUT_CSV)
else:
    print("No corpus_kriyakarama_final.json found; run parsing step first.")


Created audio manifest: c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\Data\audio\audio_manifest.csv


In [36]:
# Final verification printout
print("Check files created:")
for p in [PROCESSED / "corpus_transliterated.json", MODELS_DIR / "phoneme_map.json", MODELS_DIR / "lexicon.json", MODELS_DIR / "phoneme_index.json"]:
    print(p, "exists:", p.exists())
print("Audio folders:", (AUDIO_DIR / "words_norm").exists(), (AUDIO_DIR / "words_tts").exists())
print("Backend route file:", (ROOT / "backend" / "app" / "api" / "v1" / "routes_audio.py").exists())
print("Docs:", (DOCS_DIR / "transliteration.md").exists(), (DOCS_DIR / "audio_decision.md").exists())


Check files created:
c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\Data\processed\corpus_transliterated.json exists: True
c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\models\phoneme_map.json exists: True
c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\models\lexicon.json exists: True
c:\Users\94772\Desktop\Adaptive Audio-Based Therapeutic System for Post-Linguistic Hearing Loss Enhancing Emergency and Noisy-Environment Speech Comprehension-Test1\EarDisableResearch\Models\models\phoneme_index.json exists: True
Audio folders: True True
Backend route file: True
D